
# Evaluasi Validasi OCR (CER dan WER) pada Gambar Ber-noise dengan Filter Klasik

Notebook ini digunakan untuk mengukur dan memvalidasi performa OCR (menggunakan Tesseract OCR) dengan metrik **Character Error Rate (CER)** dan **Word Error Rate (WER)**. Pengujian ini dilakukan pada gambar teks tulisan tangan dengan berbagai jenis *noise* (seperti buram, gelap, kontras rendah, dan miring) dalam tiga kondisi gambar:
1. **Mentah (Original)**
2. **Gray (Grayscale)** setelah melalui filter prapemrosesan klasik
3. **Biner (Black & White)** setelah melalui proses binerisasi klasik

untuk data perspective sebelumnya sudah diproses dengan cara hybrid (AUTO + Manual) dalam menentukan titik sudutnya kemudian masuk filter gray dan biner ini



## Langkah 1: Ekstraksi Dataset
Upload 4 folder berikut ke dalam runtime kamu https://drive.google.com/drive/folders/1yvyCfPlgKwbY8HOwzAi-a6Ejr2yWO3mB?usp=sharing

Pada langkah ini, kita membersihkan sisa prapemrosesan sebelumnya (jika ada) dan mengekstrak berkas arsip zip dataset (`data.zip`, `gt.zip`, `out_gray.zip`, `out_bw.zip`) ke dalam direktori sementara (`tmp`).


In [9]:
!rm -rf data gt out_gray out_bw
!mkdir -p tmp
!unzip -oq data.zip -d tmp/data
!unzip -oq gt.zip -d tmp/gt
!unzip -oq out_gray.zip -d tmp/out_gray
!unzip -oq out_bw.zip -d tmp/out_bw


## Langkah 2: buat 1 berkas aja jangan dibuat nested
Di sini kita memindahkan berkas-berkas yang telah diekstrak dari folder sementara (`tmp`) ke struktur folder utama yang rapi (`data`, `gt`, `out_gray`, `out_bw`) dan menghapus direktori sementara setelah selesai.


In [10]:
import os, shutil, glob
for name in ['data','gt','out_gray','out_bw']:
    os.makedirs(name, exist_ok=True)
    for p in glob.glob(f'tmp/{name}/**/*.*', recursive=True):
        shutil.copy(p, os.path.join(name, os.path.basename(p)))
shutil.rmtree('tmp')



## Langkah 3: Verifikasi Jumlah Data
Kita memverifikasi jumlah berkas pada masing-masing folder untuk memastikan seluruh data telah diekstrak dengan lengkap (misalnya: 50 gambar data, 10 teks ground-truth/gt, 50 gambar grayscale, dan 50 gambar biner).


In [11]:
!ls data | wc -l
!ls gt | wc -l
!ls out_gray | wc -l
!ls out_bw | wc -l

50
10
50
50



## Langkah 4: Instalasi Dependensi
Langkah instalasi sistem OCR (Tesseract-OCR beserta paket bahasa Indonesia) dan pustaka Python pendukung seperti `pytesseract` untuk interface OCR, `jiwer` untuk kalkulasi nilai CER/WER, serta `opencv-python-headless` untuk manipulasi gambar.


In [12]:
!apt-get -y install tesseract-ocr tesseract-ocr-ind > /dev/null
!pip install pytesseract jiwer opencv-python-headless pandas > /dev/null


## Langkah 5: Pengujian Awal OCR & Evaluasi Tunggal
Sebelum melakukan iterasi pada seluruh dataset, kita melakukan pengujian awal menggunakan satu berkas contoh (`HW001_NORMAL.jpg`). Hasil prediksi OCR dibandingkan dengan berkas *ground truth* terkait untuk menghitung nilai CER dan WER awal.


In [13]:
import pytesseract, cv2, re, jiwer

def norm(s):
    return re.sub(r'\s+', ' ', s.lower()).strip()

img = cv2.imread('data/HW001_NORMAL.jpg')
print(img is None)   # harus False
hyp = pytesseract.image_to_string(img, lang='ind+eng')
print(hyp[:500])

gt = open('gt/HW001.txt', encoding='utf-8').read()
print('CER:', jiwer.cer(norm(gt), norm(hyp)))
print('WER:', jiwer.wer(norm(gt), norm(hyp)))

False
=| Taeg) — emvingei ada aa

| PA Dat ees APonaenuany Pantie: Penuysun Ator
| — a Them $09 arn We eo ee ere

P2. Motel Atom paves

b oda Lak enue bees

bolo Peyol Cer ey
ta SA

nde At Mm Kadar,
ae sal tokoh vial dalam
Model Ate » Membpueu Model Arwen Seperti fata Surua

eNacm)y ana ar

engclilina; atom berada pad
ara ber tentu PT
SiDU


CER: 0.6983471074380165
WER: 0.9444444444444444



## Langkah 6: Evaluasi Massal Seluruh Dataset
Bagian ini melakukan iterasi untuk memproses seluruh gambar di folder `data`. OCR dijalankan pada ketiga kondisi gambar (mentah, grayscale, dan biner) untuk setiap berkas, kemudian hasilnya dihitung nilai CER/WER-nya dibanding ground truth, dan disimpan ke dalam format CSV (`hasil_baseline.csv`).


In [14]:
import os, pandas as pd

CFG = '--psm 6'      # asumsi satu blok teks; sama untuk semua kondisi
LANG = 'ind+eng'

def ocr_path(path):
    im = cv2.imread(path)
    return pytesseract.image_to_string(im, lang=LANG, config=CFG)

def score(gt, hyp):
    g, h = norm(gt), norm(hyp)
    if not h: return 1.0, 1.0
    return jiwer.cer(g, h), jiwer.wer(g, h)

rows = []
files = sorted(os.listdir('data'))
for k, fn in enumerate(files):
    base, _ = os.path.splitext(fn)
    doc, label = base.split('_', 1)
    gt = open(f'gt/{doc}.txt', encoding='utf-8').read()

    for kondisi, path in [('mentah', f'data/{fn}'),
                          ('gray',   f'out_gray/{base}.png'),
                          ('biner',  f'out_bw/{base}.png')]:
        hyp = ocr_path(path)
        cer, wer = score(gt, hyp)
        rows.append(dict(doc=doc, noise=label.upper(), kondisi=kondisi, CER=cer, WER=wer))
    print(f'{k+1}/{len(files)} {base}')

df = pd.DataFrame(rows)
df.to_csv('hasil_baseline.csv', index=False)

1/50 HW001_BURAM
2/50 HW001_GELAP
3/50 HW001_KONTRAS RENDAH
4/50 HW001_NORMAL
5/50 HW001_PERSPEKTIF MIRING
6/50 HW002_BURAM
7/50 HW002_GELAP
8/50 HW002_KONTRAS RENDAH
9/50 HW002_NORMAL
10/50 HW002_PERSPEKTIF MIRING
11/50 HW003_BURAM
12/50 HW003_GELAP
13/50 HW003_KONTRAS RENDAH
14/50 HW003_NORMAL
15/50 HW003_PERSPEKTIF MIRING
16/50 HW004_BURAM
17/50 HW004_GELAP
18/50 HW004_KONTRAS RENDAH
19/50 HW004_NORMAL
20/50 HW004_PERSPEKTIF MIRING
21/50 HW005_BURAM
22/50 HW005_GELAP
23/50 HW005_KONTRAS RENDAH
24/50 HW005_NORMAL
25/50 HW005_PERSPEKTIF MIRING
26/50 HW006_BURAM
27/50 HW006_GELAP
28/50 HW006_KONTRAS RENDAH
29/50 HW006_NORMAL
30/50 HW006_PERSPEKTIF MIRING
31/50 HW007_BURAM
32/50 HW007_GELAP
33/50 HW007_KONTRAS RENDAH
34/50 HW007_NORMAL
35/50 HW007_PERSPEKTIF MIRING
36/50 HW008_BURAM
37/50 HW008_GELAP
38/50 HW008_KONTRAS RENDAH
39/50 HW008_NORMAL
40/50 HW008_PERSPEKTIF MIRING
41/50 HW009_BURAM
42/50 HW009_GELAP
43/50 HW009_KONTRAS RENDAH
44/50 HW009_NORMAL
45/50 HW009_PERSPEKTIF MIRING
4


## Langkah 7: Analisis Performa rata-rata CER dan WER
Terakhir, kita menyajikan ringkasan hasil berupa tabel pivot yang mengelompokkan rata-rata nilai CER dan WER berdasarkan jenis noise dan kondisi gambar untuk melihat efektivitas dari masing-masing filter prapemrosesan klasik.


In [15]:
print(df.pivot_table(index='noise', columns='kondisi', values='CER', aggfunc='mean').round(3))
print(df.pivot_table(index='noise', columns='kondisi', values='WER', aggfunc='mean').round(3))

kondisi            biner   gray  mentah
noise                                  
BURAM              0.824  2.143   0.875
GELAP              2.863  3.420   5.722
KONTRAS RENDAH     0.970  1.003   1.510
NORMAL             0.802  0.675   1.026
PERSPEKTIF MIRING  0.668  0.881   1.187
kondisi            biner   gray  mentah
noise                                  
BURAM              1.683  4.264   1.406
GELAP              6.358  7.038  11.370
KONTRAS RENDAH     2.054  2.155   2.785
NORMAL             1.558  1.290   1.941
PERSPEKTIF MIRING  1.271  1.890   2.731


In [16]:
import pandas as pd
df = pd.read_csv('hasil_baseline.csv')

for m in ['CER', 'WER']:
    print(m)
    print(df.pivot_table(index='noise', columns='kondisi', values=m, aggfunc='mean').round(3))
    print()

CER
kondisi            biner   gray  mentah
noise                                  
BURAM              0.824  2.143   0.875
GELAP              2.863  3.420   5.722
KONTRAS RENDAH     0.970  1.003   1.510
NORMAL             0.802  0.675   1.026
PERSPEKTIF MIRING  0.668  0.881   1.187

WER
kondisi            biner   gray  mentah
noise                                  
BURAM              1.683  4.264   1.406
GELAP              6.358  7.038  11.370
KONTRAS RENDAH     2.054  2.155   2.785
NORMAL             1.558  1.290   1.941
PERSPEKTIF MIRING  1.271  1.890   2.731

